# 02 · Silver — Tipado, validacion y cuarentena

| | |
|---|---|
| **Objetivo** | Convertir el dato crudo en dato confiable, aislando lo que no cumple las reglas |
| **Entradas** | `bronze.raw_trips` |
| **Salidas** | `silver.clean_trips`, `silver.quarantine_trips`, `silver.dq_metrics` |
| **Depende de** | Notebook 01 ejecutado con `p_origen=train` |

**Proceso**
1. Tipar las columnas y verificar que ningun cast falle en silencio
2. Deduplicar por hash de fila
3. Evaluar cinco reglas de calidad, cada una como bandera independiente
4. Registrar las metricas de calidad por regla
5. Separar entre tabla limpia y cuarentena

**Principio de la capa:** no se borra el dato invalido, se aisla. Borrar
oculta el problema; derivar a cuarentena lo hace contable y trazable.

La logica reside en `src/nyc_taxi/data_prep/`; este notebook orquesta y
documenta, de modo que las conversiones y las reglas queden cubiertas por
pruebas automaticas.

In [0]:
import os
import sys


# Localiza la raiz del repo subiendo hasta encontrar src/, en vez de fijar un
# numero de saltos. Asi el notebook funciona a cualquier profundidad.
def _preparar_path(marcador="src", max_niveles=10):
    ruta = os.getcwd()
    for _ in range(max_niveles):
        if os.path.isdir(os.path.join(ruta, marcador)):
            destino = os.path.join(ruta, marcador)
            if destino not in sys.path:
                sys.path.insert(0, destino)
            return destino
        padre = os.path.dirname(ruta)
        if padre == ruta:
            break
        ruta = padre
    raise RuntimeError(f"No se encontro la raiz del repo (carpeta con {marcador}/)")


_preparar_path()

from pyspark.sql import functions as F

from nyc_taxi import config
from nyc_taxi.data_prep import typing as tp
from nyc_taxi.data_prep import validation

In [0]:
df_bronze = spark.table(config.TBL_RAW_TRIPS)
n_inicial = df_bronze.count()
print(f"Filas en Bronze: {n_inicial:,}")

Filas en Bronze: 1,458,644


## 1. Tipado

Ver `data_prep/typing.py` para el detalle de cada conversión. Las dos
decisiones que no son obvias: las coordenadas van a `double` y no a
`float`, porque 32 bits no alcanzan para la precisión del dato original y
el redondeo se traduce en decenas de metros; y los identificadores se
quedan en texto porque no son cantidades.

**Un cast fallido en Spark no lanza excepción: produce nulos en silencio.**
Por eso se compara el conteo de nulos antes y después de convertir.

In [0]:
cols_negocio = [c for c in df_bronze.columns if not c.startswith("_")]

nulos_antes = df_bronze.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols_negocio]
).first().asDict()

df = tp.tipar_trips(df_bronze)

nulos_despues = df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols_negocio]
).first().asDict()

print(f"{'columna':<22} {'tipo':<12} {'nulos antes':>12} {'nulos despues':>14}")
for c, t in df.dtypes:
    if c in cols_negocio:
        print(f"{c:<22} {t:<12} {nulos_antes[c]:>12,} {nulos_despues[c]:>14,}")

nuevos_nulos = {c: nulos_despues[c] - nulos_antes[c] for c in cols_negocio
                if nulos_despues[c] > nulos_antes[c]}
assert not nuevos_nulos, f"El casteo introdujo nulos: {nuevos_nulos}"
print("\nNingun cast fallo silenciosamente.")

columna                tipo          nulos antes  nulos despues
id                     string                  0              0
vendor_id              string                  0              0
pickup_datetime        timestamp               0              0
dropoff_datetime       timestamp               0              0
passenger_count        int                     0              0
pickup_longitude       double                  0              0
pickup_latitude        double                  0              0
dropoff_longitude      double                  0              0
dropoff_latitude       double                  0              0
store_and_fwd_flag     int                     0              0
trip_duration          int                     0              0

Ningun cast fallo silenciosamente.


## 2. Deduplicación

Se usa el `_row_hash` calculado en Bronze, que cubre todas las columnas de
negocio. Bronze ya reportó cero duplicados exactos y cero ids repetidos,
así que esta regla no va a eliminar nada: queda operando de forma
preventiva, para el caso de una reingesta o de un archivo reenviado.

In [0]:
df = df.dropDuplicates(["_row_hash"])
n_tras_dedup = df.count()
print(f"Filas tras deduplicar: {n_tras_dedup:,}  (eliminadas: {n_inicial - n_tras_dedup:,})")

Filas tras deduplicar: 1,458,644  (eliminadas: 0)


## 3. Validaciones

Cada regla se materializa como una columna booleana propia en vez de
encadenar filtros. Eso permite contar cuántas filas falla **cada regla por
separado**, en lugar de saber solo que algo falló, y es lo que hace
posible detectar un umbral mal calibrado: si una sola regla rechaza un
porcentaje alto, el sospechoso es el umbral y no el dato.

Los umbrales y su justificación están en `config.py`.

In [0]:
reglas = validation.construir_reglas_spark()

for nombre, expr in reglas.items():
    df = df.withColumn(nombre, expr)

df = df.withColumn("es_valido", F.expr(" AND ".join(validation.NOMBRES_REGLAS)))

# El cómputo serverless no soporta cache(). Sin persistir de alguna forma, el
# DataFrame se recomputaría en cada acción: una vez para el resumen de calidad y
# otra por cada tabla escrita. Materializar en una tabla Delta intermedia logra
# el mismo objetivo: se escribe una vez y las lecturas posteriores no vuelven a
# evaluar las expresiones de validación. El guion bajo marca que es intermedia.
TBL_VALIDADO = f"{config.CATALOGO}.{config.SCHEMA_SILVER}._trips_validado"

(df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(TBL_VALIDADO))

df = spark.table(TBL_VALIDADO)

## 4. Métricas de calidad

Una sola pasada agregada en vez de un `count()` por regla: cinco conteos
separados serían cinco escaneos completos de la tabla.

In [0]:
agregados = [
    F.sum((~F.col(r)).cast("int")).alias(r) for r in validation.NOMBRES_REGLAS
] + [
    F.sum((~F.col("es_valido")).cast("int")).alias("total_invalidas"),
    F.count(F.lit(1)).alias("total_filas"),
]

resumen = df.select(agregados).first().asDict()

total = resumen["total_filas"]
print(f"{'regla':<20} {'fallos':>10} {'% del total':>12}")
for r in validation.NOMBRES_REGLAS:
    print(f"{r:<20} {resumen[r]:>10,} {resumen[r] / total * 100:>11.3f}%")
print("-" * 44)
print(f"{'total invalidas':<20} {resumen['total_invalidas']:>10,} "
      f"{resumen['total_invalidas'] / total * 100:>11.3f}%")

regla                    fallos  % del total
v_duracion                2,061       0.141%
v_pasajeros                  65       0.004%
v_coords_pickup             256       0.018%
v_coords_dropoff            985       0.068%
v_completitud                 0       0.000%
--------------------------------------------
total invalidas           3,118       0.214%


Los conteos se persisten en `dq_metrics` con marca de tiempo. Esa tabla es
la evidencia citable del resumen ejecutivo y, acumulada entre corridas, el
insumo natural del monitoreo de calidad en producción.

In [0]:
filas_dq = [
    (regla, int(resumen[regla]), int(total), float(resumen[regla] / total * 100))
    for regla in validation.NOMBRES_REGLAS
]

df_dq = (
    spark.createDataFrame(filas_dq, ["regla", "filas_fallidas", "filas_evaluadas", "pct_fallo"])
    .withColumn("tabla_origen", F.lit(config.TBL_RAW_TRIPS))
    .withColumn("evaluado_en", F.current_timestamp())
)

df_dq.write.format("delta").mode("append").saveAsTable(config.TBL_DQ_METRICS)
display(df_dq)

regla,filas_fallidas,filas_evaluadas,pct_fallo,tabla_origen,evaluado_en
v_duracion,2061,1458644,0.1412956142828545,nyc_taxi.bronze.raw_trips,2026-08-10T04:56:46.463Z
v_pasajeros,65,1458644,0.004456193560594635,nyc_taxi.bronze.raw_trips,2026-08-10T04:56:46.463Z
v_coords_pickup,256,1458644,0.017550546946341946,nyc_taxi.bronze.raw_trips,2026-08-10T04:56:46.463Z
v_coords_dropoff,985,1458644,0.067528471649011,nyc_taxi.bronze.raw_trips,2026-08-10T04:56:46.463Z
v_completitud,0,1458644,0.0,nyc_taxi.bronze.raw_trips,2026-08-10T04:56:46.463Z


## 5. Separación

Las banderas por regla se conservan en cuarentena —son el motivo del
rechazo— y se descartan de la tabla limpia, donde ya no aportan nada
porque todas valen verdadero por construcción.

In [0]:
cols_finales = [c for c in df.columns if c not in validation.NOMBRES_REGLAS + ["es_valido"]]

df_clean = df.filter("es_valido").select(*cols_finales)
df_quarantine = df.filter("NOT es_valido")

(df_clean.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(config.TBL_CLEAN))

(df_quarantine.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(config.TBL_QUARANTINE))

n_clean = spark.table(config.TBL_CLEAN).count()
n_quar = spark.table(config.TBL_QUARANTINE).count()

print(f"clean_trips      : {n_clean:,}")
print(f"quarantine_trips : {n_quar:,}")
print(f"retencion        : {n_clean / n_inicial * 100:.2f}%")

assert n_clean + n_quar == n_tras_dedup, "Se perdieron filas en la separacion"

# La tabla intermedia ya cumplió su función.
spark.sql(f"DROP TABLE IF EXISTS {TBL_VALIDADO}")

clean_trips      : 1,455,526
quarantine_trips : 3,118
retencion        : 99.79%


DataFrame[]

## Inspección de lo rechazado

Mirar la cuarentena no es opcional: es lo que distingue un dato realmente
inválido de un umbral mal puesto. Si un grupo grande de filas cae por una
sola regla y a simple vista parecen viajes legítimos, hay que revisar el
criterio antes que el dato.

La primera tabla muestra las combinaciones de reglas que fallan juntas,
que suele revelar el patrón: coordenadas en (0,0) con duración absurda
apunta a un GPS sin señal, no a dos problemas independientes.

In [0]:
display(
    spark.table(config.TBL_QUARANTINE)
    .groupBy(*validation.NOMBRES_REGLAS)
    .count()
    .orderBy(F.desc("count"))
)

v_duracion,v_pasajeros,v_coords_pickup,v_coords_dropoff,v_completitud,count
false,true,true,true,true,2056
true,true,true,false,true,739
true,true,false,false,true,242
true,false,true,true,true,63
true,true,false,true,true,12
false,true,true,false,true,3
false,false,true,true,true,1
true,false,false,false,true,1
false,true,false,true,true,1


In [0]:
display(
    spark.table(config.TBL_QUARANTINE)
    .select("id", "pickup_datetime", "trip_duration", "passenger_count",
            "pickup_latitude", "pickup_longitude", *validation.NOMBRES_REGLAS)
    .limit(20)
)

id,pickup_datetime,trip_duration,passenger_count,pickup_latitude,pickup_longitude,v_duracion,v_pasajeros,v_coords_pickup,v_coords_dropoff,v_completitud
id0412148,2016-04-29T20:39:18.000Z,3,1,40.66777038574219,-73.5481185913086,true,true,false,false,true
id3792610,2016-02-06T13:17:20.000Z,84985,1,40.75859069824219,-73.97746276855469,false,true,true,true,true
id2348143,2016-02-28T04:46:46.000Z,1795,1,40.75949478149414,-73.2226791381836,true,true,false,false,true
id3385534,2016-05-28T17:14:15.000Z,24345,5,40.64516067504883,-73.77682495117188,false,true,true,true,true
id1380296,2016-05-08T23:01:13.000Z,86352,6,40.751991271972656,-73.9935531616211,false,true,true,true,true
id1082420,2016-04-01T07:21:32.000Z,86335,1,40.750877380371094,-73.97216796875,false,true,true,true,true
id3146038,2016-06-05T16:57:32.000Z,7134,1,40.77402877807617,-73.87306213378906,true,true,true,false,true
id1349848,2016-03-14T09:41:23.000Z,1156,2,40.647308349609375,-73.7901382446289,true,true,true,false,true
id3967786,2016-01-17T13:40:55.000Z,86356,2,40.7280387878418,-74.00196838378906,false,true,true,true,true
id0784573,2016-03-28T17:55:13.000Z,782,2,36.029300689697266,-77.44075012207031,true,true,false,false,true
